# Transfer Learning, Residual Networks, and Adversarial Examples

In the previous notebook, we built a CNN from basic components: convolution, activation, pooling, and fully connected layers.

In this notebook, we move to more modern computer vision topics:

1. **Residual networks** — how skip connections help train deeper CNNs
2. **Transfer learning** — how pretrained CNNs can be reused for new tasks
3. **Fine-tuning** — deciding which layers to freeze or train
4. **Adversarial examples** — small input changes that can fool neural networks
5. **Adversarial training** — improving robustness by training on perturbed examples

The goal is not only to train models, but also to understand important design choices and weaknesses of modern CNNs.

## 1. Residual Networks: Residual Connections and ResNet Blocks

Very deep neural networks are difficult to train.
As more layers are added, gradients can become unstable or very small, making optimization harder.

Residual networks address this problem using **skip connections**.

Instead of learning only:

$$
out = F(x)
$$

the block computes:

$$
out = F(x) + x
$$

where \(x\) is the original input and \(F(x)\) is the result of several convolutional layers.

This is called a **skip connection** because the input skips over some layers and is added back later.

Residual connections help deep networks because:

- useful information can pass through the block more easily
- gradients can flow more directly during backpropagation
- the block can learn small corrections instead of a completely new representation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### Task 1: Implement Residual Block

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        identity = x

        # TODO: apply first convolution and ReLU
        out = ...

        # TODO: apply second convolution
        out = ...

        # TODO: add skip connection
        out = ...

        # TODO: apply final ReLU
        out = ...

        return out

Test your implementation, a residual block must return the same shape if input and output are added.

In [ ]:
x = torch.randn(4, 16, 28, 28)

block = ResidualBlock(channels=16)
out = block(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

**Question:**
Why must `x` and `out` have the same shape before they are added?


**Answers:**



## 2. Transfer Learning with a Pretrained CNN

Training large CNNs from scratch requires a lot of data and computation.

Transfer learning uses a model that was already trained on a large dataset, such as ImageNet, and adapts it to a new task.

A common strategy is:

1. load a pretrained model
2. freeze the feature extractor
3. replace the final classification layer
4. train only the new classifier first

Here we use ResNet-18, a smaller residual network available in `torchvision`.

In [ ]:
from torchvision import models
from torchvision.models import ResNet18_Weights

In [ ]:
weights = ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

print(model)

Let's inspect the final classifier:

In [ ]:
print(model.fc)

The original ResNet-18 model predicts 1000 ImageNet classes.
For a new dataset, we need to replace this final layer with a classifier that has the correct number of output classes.

### Task 2: Freeze backbone and replace classifier

In [ ]:
num_classes = 10

# TODO: freeze all pretrained parameters
for param in model.parameters():
    ...

# TODO: get number of input features of the old classifier
num_features = ...

# TODO: replace the final classification layer
model.fc = ...

print(model.fc)

In [ ]:
# Let's check trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

## 3. Training the New Classifier

After replacing the final layer, only the new classifier is trainable.

The pretrained ResNet acts as a fixed feature extractor:

$$
image \rightarrow pretrained\ features \rightarrow new\ classifier
$$

In this section, we train only the new final layer on CIFAR-10.

In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader, Subset

In [ ]:
# Use the preprocessing expected by the pretrained ResNet
preprocess = weights.transforms()

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=preprocess
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=preprocess
)

# Small subsets to keep training fast
generator = torch.Generator().manual_seed(42)

train_indices = torch.randperm(len(train_dataset), generator=generator)[:2000]
test_indices = torch.randperm(len(test_dataset), generator=generator)[:500]

train_subset = Subset(train_dataset, train_indices)
test_subset = Subset(test_dataset, test_indices)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
print(class_names)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

criterion = nn.CrossEntropyLoss()

# Only train parameters with requires_grad=True
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

### Task 3: Implement training loop

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        # TODO: reset gradients
        ...

        # TODO: forward pass
        outputs = ...

        # TODO: compute loss
        loss = ...

        # TODO: backward pass
        ...

        # TODO: update parameters
        ...

        total_loss += loss.item()

        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    test_loss, test_acc = evaluate(
        model, test_loader, criterion, device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}"
    )

## 4. Fine-Tuning the Last Layers

In the previous section, ResNet was used as a fixed feature extractor.
Only the new classification layer was trained.

This is often a good first step, but sometimes the pretrained features are not perfectly suited to the new dataset.

Fine-tuning means that we unfreeze some pretrained layers and continue training them with a smaller learning rate.

Here, we unfreeze only the last ResNet block, `layer4`, while keeping the earlier layers frozen.

In [ ]:
# First, keep all parameters frozen
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the final classification layer
for param in model.fc.parameters():
    param.requires_grad = True

# Unfreeze the last ResNet block
for param in model.layer4.parameters():
    param.requires_grad = True

In [ ]:
# Let's check again what's trainable
trainable_params = 0
total_params = 0

for name, param in model.named_parameters():
    total_params += param.numel()

    if param.requires_grad:
        trainable_params += param.numel()
        print(name)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

### Task 4: Create an optimizer that updates only trainable parameters

In [ ]:
# TODO: create an optimizer that updates only trainable parameters with smaller learning rate (0.0001)
optimizer = torch.optim.Adam(
    ...,
    ...
)

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    test_loss, test_acc = evaluate(
        model, test_loader, criterion, device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}"
    )

**Questions**

1. Why do we use a smaller learning rate for fine-tuning?
2. Why might fine-tuning improve performance compared to training only the classifier?
3. Why might fine-tuning be risky on a very small dataset?

**Answers:**



## 5. Adversarial Examples with FGSM

Neural networks can be sensitive to very small changes in the input image.

An **adversarial example** is an input that looks almost unchanged to humans, but causes the model to make a wrong prediction.

The Fast Gradient Sign Method, or **FGSM**, creates such an image by changing the input in the direction that increases the loss:

$$
x_{adv} = x + \epsilon \cdot sign(\nabla_x J(\theta, x, y))
$$

where:

- $x$ is the original image
- $y$ is the true label
- $J$ is the loss function
- $\epsilon$ controls the perturbation size

In [ ]:
model.eval()

images, labels = next(iter(test_loader))

image = images[0:1].to(device)
label = labels[0:1].to(device)

image.requires_grad = True

In [ ]:
output = model(image)
prediction = torch.argmax(output, dim=1)

print("True label:", class_names[label.item()])
print("Predicted label:", class_names[prediction.item()])

### Task 5: Implemept FGSM attack

In [ ]:
def fgsm_attack(image, epsilon, gradient):
    """
    Create an adversarial image using FGSM.

    Args:
        image: normalized input image
        epsilon: perturbation strength
        gradient: gradient of the loss with respect to the image

    Returns:
        adversarial image
    """

    # TODO: get the sign of the gradient
    signed_gradient = ...

    # TODO: add perturbation to the image
    adversarial_image = ...

    # ImageNet normalization constants
    mean = torch.tensor([0.485, 0.456, 0.406], device=image.device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=image.device).view(1, 3, 1, 1)

    # TODO: calculate lower and upper bounds for normalized images
    lower_bound = ...
    upper_bound = ...

    # TODO: clip adversarial image to valid normalized image range
    adversarial_image = ...

    return adversarial_image

In [ ]:
images, labels = next(iter(test_loader))

image = images[0:1].to(device).detach().clone()
label = labels[0:1].to(device)

image.requires_grad_(True)

output = model(image)
loss = criterion(output, label)

model.zero_grad()
loss.backward()

image_gradient = image.grad.data

epsilon = 0.03

adversarial_image = fgsm_attack(
    image,
    epsilon,
    image_gradient
)

In [ ]:
adversarial_output = model(adversarial_image)
adversarial_prediction = torch.argmax(adversarial_output, dim=1)

print("Original prediction:", class_names[prediction.item()])
print("Adversarial prediction:", class_names[adversarial_prediction.item()])

In [ ]:
import matplotlib.pyplot as plt

imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def unnormalize(image_tensor):
    image = image_tensor.detach().cpu().clone()
    image = image * imagenet_std + imagenet_mean
    image = torch.clamp(image, 0, 1)
    return image

In [ ]:
original_image = unnormalize(image).squeeze(0).permute(1, 2, 0)
adv_image = unnormalize(adversarial_image).squeeze(0).permute(1, 2, 0)

perturbation = adv_image - original_image

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original_image)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(torch.clamp(perturbation * 10 + 0.5, 0, 1))
axes[1].set_title("Perturbation ×10")
axes[1].axis("off")

axes[2].imshow(adv_image)
axes[2].set_title("Adversarial Image")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 6. Adversarial Training

One simple defense against adversarial examples is to include adversarial examples during training.

Instead of training only on clean images, we train on both:

- original images
- adversarially perturbed images

This can improve robustness, but it may also make training slower or reduce clean accuracy.

In [ ]:
def adversarial_train_one_epoch(model, dataloader, criterion, optimizer, device, epsilon):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        # Create adversarial images
        images_adv = images.detach().clone()
        images_adv.requires_grad_(True)

        outputs_adv = model(images_adv)
        loss_adv = criterion(outputs_adv, labels)

        model.zero_grad()
        loss_adv.backward()

        gradients = images_adv.grad.data

        # TODO: create adversarial images using FGSM
        images_adv = ...

        # Train on clean and adversarial images
        combined_images = torch.cat([images, images_adv], dim=0)
        combined_labels = torch.cat([labels, labels], dim=0)

        # TODO: reset gradients
        ...

        # TODO: forward pass
        outputs = ...

        # TODO: compute loss
        loss = ...

        # TODO: backward pass
        ...

        # TODO: update parameters
        ...

        total_loss += loss.item()

        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == combined_labels).sum().item()
        total += combined_labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
epsilon = 0.03
num_epochs = 2

for epoch in range(num_epochs):
    train_loss, train_acc = adversarial_train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
        epsilon
    )

    test_loss, test_acc = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}"
    )

**Questions**

1. Why does adversarial training take longer than normal training?
2. Why might adversarial training improve robustness?
3. Why might clean test accuracy sometimes decrease after adversarial training?

**Answers:**



---
#### Feedback

To help improve the exercises, please take 2 minutes to fill out an
[anonymous questionnaire](https://survey.mbp.tf.uni-bielefeld.de/index.php?r=survey/index&sid=888254).
